# Literature Review Agent
### Description
- Building a multiple agentic system for help researchers to save some time in literature. 

## Agents and their work
|Agents & Tool|Work description|
|------------|---------------|
|Search_Agent| The agent search research paper to the user's topic|
|Search_Tool| The tools help the agent to search the paper using the keywords|
|Downloader| The tools used to download those searched papers|
|DB_Agent| The agent read and divide into data and save those in a vector database|
|Question_Agent| THe agent generate research questions|
|Answer_agent| The agent answer those research question|
|Synthesis_Agent| Finalize the review|

### Dependencies
```bash
pip install langgraph langchain-groq langchain-openrouter chromadb \
            pypdf arxiv duckduckgo-search streamlit python-dotenv
```



## Agent : 1 - Search agent
- This agent will get topic from users input and find keywords and start searching related 

In [18]:
# Calling LLM and setup API
import os
import langchain_openrouter
import langchain_groq
from dotenv import load_dotenv

# Run API key from .env file
load_dotenv()
GROQ_API = os.getenv("GROQ_API")
OPEN_ROUTER_API = os.getenv("OPEN_ROUTER_API")

if not GROQ_API:
    raise ValueError("API keys for GROQ must be set in the .env file.")
else:
    print("GROQ API key is set.")
if not OPEN_ROUTER_API:
    raise ValueError("API keys for OPEN_ROUTER must be set in the .env file.")
else:
    print("OPEN_ROUTER API key is set.")


GROQ API key is set.
OPEN_ROUTER API key is set.


In [19]:
from langchain_groq import ChatGroq
from langchain_openrouter import ChatOpenRouter
# Search agent
llm_search = ChatGroq(model="openai/gpt-oss-20b", 
                      api_key=GROQ_API, 
                      temperature=0, 
                      max_tokens=1000)
# Testing
response = llm_search.invoke("What is Tuberculosis?")
print(f"LLM Read for search agent with model: {llm_search.model}")
print(f"Response: {response}")



LLM Read for search agent with model: openai/gpt-oss-20b
Response: content='**Tuberculosis (TB)** is a contagious bacterial infection caused by *Mycobacterium tuberculosis*. It most commonly affects the lungs (pulmonary TB) but can involve any organ (extrapulmonary TB).\n\n| Aspect | Key Points |\n|--------|------------|\n| **Transmission** | Airborne droplets from a person with active pulmonary TB who coughs, sneezes, or speaks. |\n| **Incubation** | Weeks to months after exposure. |\n| **Latent vs. Active** | *Latent TB infection* (LTBI) – bacteria are present but inactive; no symptoms, not contagious. *Active TB* – symptoms, contagious. |\n| **Symptoms (pulmonary)** | Persistent cough (≥3\u202fweeks), chest pain, sputum that may be bloody, fever, night sweats, weight loss, fatigue. |\n| **Risk Factors** | HIV infection, diabetes, mal' additional_kwargs={'reasoning_content': 'The user asks: "What is Tuberculosis?" They likely want a concise explanation. Provide definition, cause, tra

In [20]:
# Search query & tool implementation
import requests
import feedparser
def generate_search_queries(topic, n=3):
    prompt = f"""You are a research assistant. Generate {n} search queries for the topic: "{topic}".
    Each query should be concise and relevant to the topic. Return the queries as a list of strings.
    Return only numbered list of queries without any additional text or explanation.
    
    Topic:{topic}
    """
    response = llm_search.invoke(prompt) 
    lines = [line.strip() for line in response.content.split("\n") if line.strip()]
    queries =[]
    for line in lines:
        if line[0].isdigit():
            q = line.split('.',1)[1].strip()
            queries.append(q)
        return queries[:n] if queries else [topic]
    
def search_arxiv(query, max_results=5):
    base_url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": max_results,
        "sortBy": "relevance",
        "sortOrder": "descending"
    }
    try:
        response = requests.get(base_url, params=params, timeout=20)
        feed = feedparser.parse(response.text)
    except Exception as e:
        print(f"arxiv search failed for query '{query}': {e}")
        return []
    
    results = []
    for entry in feed.entries:
        result = {
            "title": entry.title.replace('\n', ' ').strip(), # Remove newlines and extra spaces
            "authors":[a.name for a in entry.authors],
            "abstract": entry.summary.replace('\n', ' ').strip(), 
            "pdf_url": entry.published,
            "source": "arxiv"
        }
        results.append(result)
    return results
def search_semantic_scholar(query, max_results=5):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": query,
        "limit": max_results,
        "fields": "title,abstract,authors,url,openAccessPdf,year"
    }
    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()
    except Exception as e:
        print(f"Semantic Scholar search failed for '{query}': {e}")
        return []

    results = []
    for paper in data.get("data", []):
        results.append({
            "title": paper.get("title"),
            "authors": [a["name"] for a in paper.get("authors", [])],
            "abstract": paper.get("abstract") or "",
            "pdf_url": paper.get("openAccessPdf", {}).get("url") if paper.get("openAccessPdf") else None,
            "published": paper.get("year"),
            "source": "semantic_scholar"
        })
    return results


    



In [21]:
# Search agent
def search_agent(topic, max_results_per_query=5,display_results=True):
    queries = generate_search_queries(topic)
    print(f"Generated queries: {queries}")
    
    all_results = []
    for q in queries:
        all_results.extend(search_arxiv(q, max_results_per_query))
        all_results.extend(search_semantic_scholar(q, max_results_per_query))
    
    seen = set()
    unique_results = []
    for r in all_results:
        if not r.get("title"):
            continue
        key = r["title"].lower().strip()
        if key not in seen:
            seen.add(key)
            unique_results.append(r)
    print(f"Found {len(unique_results)} unique papers")
    
    if display_results and unique_results:
        print("\n Search results:")
        print("=" * 60)
        for i, paper in enumerate(unique_results, 1):
            title = paper.get("title","No Title")
            authors = paper.get("authors",["Unknows"])[:2] # First 2 authors only
            year = paper.get("Year","N/A")
            
            print(f"{i:2}. Title: {title}")
            print(f"    Authors: {', '.join(authors) if authors else 'Unknown'}")
            print(f"    Year: {year}")
            print("-" * 60)
    return unique_results

if __name__ == "__main__":
    user_topic = "Tuberculosis detection by AI using chest X-ray images"
    max_papers = 5
    # user_topic = input("Enter a research topic: ")
    # max_papers = int(input("Enter the maximum number of papers to retrieve per query: "))
    papers = search_agent(user_topic, max_results_per_query=max_papers, display_results=True)

Generated queries: ['AI-based tuberculosis detection from chest X-ray images']
Found 5 unique papers

 Search results:
 1. Title: Classification of Pneumonia and Tuberculosis from Chest X-rays
    Authors: M. Abubakar, I. Shah
    Year: N/A
------------------------------------------------------------
 2. Title: An Efficient Mixture of Deep and Machine Learning Models for COVID-19 and Tuberculosis Detection Using X-Ray Images in Resource Limited Settings
    Authors: Ali H. Al-Timemy, Rami N. Khushaba
    Year: N/A
------------------------------------------------------------
 3. Title: Few-Shot Learning Approach on Tuberculosis Classification Based on Chest X-Ray Images
    Authors: A. A. G. Yogi Pramana, Faiz Ihza Permana
    Year: N/A
------------------------------------------------------------
 4. Title: Reliable Tuberculosis Detection using Chest X-ray with Deep Learning, Segmentation and Visualization
    Authors: Tawsifur Rahman, Amith Khandakar
    Year: N/A
---------------------

## Phase 2 : Download and analyse those founded papers
### Analysing agent

In [22]:
# Download function
import os
import requests

def download_pdf(pdf_url, save_dir="data/papers",filename=None):
    if not pdf_url:
        print("No PDF URL provided.")
        return None
    os.makedirs(save_dir, exist_ok=True)
    filename = filename or pdf_url.split("/")[-1].replace(".pdf","") + ".pdf"
    filepath = os.path.join(save_dir, filename)
    
    if os.path.exists(filepath):
        print(f"Download skipped: {filename} already exists.")
        return filepath
    try:
        response = requests.get(pdf_url, timeout=30)
        if response.status_code == 200 and response.headers.get("content-type", "").startswith("application/pdf"):
            with open(filepath, "wb") as f:
                f.write(response.content)
            print(f"Downloaded: {filename}")
            return filepath
        else:
            print(f"Failed to download {filename}. Status code: {response.status_code}, Content-Type: {response.headers.get('content-type')}")
            return None
    except Exception as e:
        print(f"Error downloading {filename}: {e}")
        return None

### Extract text from pdf

In [23]:
from pypdf import PdfReader

def extract_text(filepath):
    try:
        reader = PdfReader(filepath)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text.strip()
    except Exception as e:
        print(f"Error extracting text from {filepath}: {e}")
        return ""

In [24]:
# Relevence filter
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()
GROQ_API = os.getenv("GROQ_API")
llm_relevance = ChatGroq(model="openai/gpt-oss-20b", api_key=GROQ_API, temperature=0, max_tokens=1000)

def is_relevant(abstract, topic):
    if not abstract:
        return False
    prompt = f"""Topic: {topic}
    Abstract: {abstract}
    Is this paper relavant to the topic? Answer with 'Yes' or 'No' only.
    """
    response = llm_relevance.invoke(prompt)
    return "yes" in response.content.lower().strip()

In [25]:
# Chunk test
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_text(text, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(text)

In [26]:
import torch
print(torch.cuda.is_available())

True


In [27]:
# Embed + store in chroma
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="data/vector_db")
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")

collection = client.get_or_create_collection(name="literature_review", embedding_function=embedding_fn)

def store_chunks(chunks, paper_meta, chunk_id_prefix):
    ids = [f"{chunk_id_prefix}_{i}" for i in range(len(chunks))]
    metadatas = [{"title": paper_meta["title"], "source": paper_meta["source"]} for _ in chunks]
    collection.add(documents=chunks, ids=ids, metadatas=metadatas)
    

In [28]:
# Analysis agent
def analysis_agent(papers, topic):
    stored_count=0
    for idx, paper in enumerate(papers):
        if not is_relevant(paper.get("Abstract"), topic):
            print(f"Paper {idx+1} is not relevant. Skipping.")
            continue
        
        filepath = download_pdf(paper.get("pdf_url"), filename=f"paper_{idx+1}.pdf")
        if not filepath:
            print(f"Skipped (no pdf): {paper.get('title')}")
            continue
        
        text = extract_text(filepath)
        if not text:
            print(f"Skipped (no text): {paper.get('title')}")
            continue
        
        chunks = chunk_text(text)
        store_chunks(chunks, paper, chunk_id_prefix=f"paper_{idx}")
        stored_count +=1
        print(f"Stored: {paper['title']} ({len(chunks)} chunks)")
        
    print(f"\n {stored_count} papers stored in the vector database.")

In [ ]:
topic = "cross-generator generalization AI-Generated text detection"
papers = search_agent(topic, max_results_per_query=5, display_results=True)
analysis_agent(papers, topic)

Generated queries: ['cross-generator generalization AI text detection']
Found 100 unique papers

 Search results:
 1. Title: Sarang at DEFACTIFY 4.0: Detecting AI-Generated Text Using Noised Data and an Ensemble of DeBERTa Models
    Authors: Avinash Trivedi, Sangeetha Sivanesan
    Year: N/A
------------------------------------------------------------
 2. Title: Faith in AI can narrow the futures individuals consider
    Authors: Aoi Naito, Hirokazu Shirado
    Year: N/A
------------------------------------------------------------
 3. Title: Foundations of GenIR
    Authors: Qingyao Ai, Jingtao Zhan
    Year: N/A
------------------------------------------------------------
 4. Title: Multi-Hierarchical Feature Detection for Large Language Model Generated Text
    Authors: Luyan Zhang, Xinyu Xie
    Year: N/A
------------------------------------------------------------
 5. Title: mdok of KInIT: Robustly Fine-tuned LLM for Binary and Multiclass AI-Generated Text Detection
    Authors: D